# Explore VPU Kernels

Goal: print the lowered IR graph for simple ops to understand what VPU kernels look like.

Pipeline:
```
nn.Module → tracing.trace() → ir.Graph (frontend) → sdk.convert() → ir.Graph (with VpuKernels)
```

Start with `reduce_sum` (simplest), then `layernorm` (more complex).

## Step 1 — Build a tiny nn.Module using reduce_sum

In [1]:
import tempfile
import torch
from safetensors.torch import save_file

import tensordyne.nn as nn
import tensordyne.ir as ir
from tensordyne.nn import tracing
import tensordyne.sdk as sdk


class ReduceSumModel(nn.Module):
    def __init__(self):
        super().__init__()

    def build(self, x: nn.Tensor) -> nn.Tensor:
        return nn.reduce_sum(x, axis=-1, keep_dim=True)


model = ReduceSumModel()
print(model)

ReduceSumModel()


## Step 2 — Trace the model into an `ir.Graph`

Tracing converts the `nn.Module` into a frontend IR graph. At this point there are NO VpuKernels yet — just frontend instructions like `ReduceSum`.

In [2]:
# IMPORTANT: VPU only supports Rfp16 / Int16 / UInt16.
# Using Fp32 gives a GENERIC (non-VPU) lowering — you'll see 0 VpuKernels.
# Rfp16(-15) means 16-bit float with exponent bias = -15 (the standard VPU dtype).
with tempfile.TemporaryDirectory() as tmp:
    graph = tracing.trace(
        model,
        (nn.Tensor(shape=(4, 32), dtype=ir.Rfp16(-15)),),
        constants_path=tmp,
    )

print("=== FRONTEND GRAPH (before sdk.convert) ===")
print(graph)

=== FRONTEND GRAPH (before sdk.convert) ===
main = RootGraph()
    input_0 = Rfp16:eb(-15){4, 32}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (4, 32)), on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))()
    _T2 = Rfp16:eb(-15){4, 1}:AMEM << ReduceSum(axis=1, output_dtype=Rfp16(eb=-15))(input_0):runOn(chip=0, cluster=0)
    output_0 = Rfp16:eb(-15){4, 1}:AMEM << compiler.Output(on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))(_T2)


## Step 3 — Compile the graph to actually lower to VPU kernels

**Important correction:** `sdk.convert()` only sets dtypes — it does NOT lower to VPU kernels.
The actual lowering happens in `sdk.compile()` (which calls `convert` internally first).

We can either:
1. Run the full `sdk.compile()` pipeline (produces a `.tdx` binary)
2. Inspect the `debug_artifacts_path` output to see intermediate IR graphs


In [3]:
from tensordyne.compiler._bindings.compiler import Target
import tempfile
from pathlib import Path

# First convert (sets dtypes)
conversion_output = sdk.convert(graph)

# Then compile — this is where lowering to VPU kernels happens.
# debug_artifacts_path saves intermediate IR snapshots so we can read them.
tmp = tempfile.mkdtemp()
tdx_path = Path(tmp) / 'reduce_sum.tdx'
debug_path = Path(tmp) / 'debug'

compilation_output = sdk.compile(
    graph=conversion_output.graph,
    conversion_config=conversion_output.conversion_config,
    target=Target.Pyxis,
    tdx_path=tdx_path,
    debug_artifacts_path=debug_path,
    verbose=False,
)

print(f'CompilationOutput attrs: {[a for a in dir(compilation_output) if not a.startswith("_")]}')
print(f'\nDebug artifacts written to: {debug_path}')
for p in sorted(Path(debug_path).rglob("*")):
    if p.is_file():
        print(f'  {p.relative_to(debug_path)}')


Skipping op _T2 (_T2 = Rfp16:eb(-15){4, 1}:AMEM << ReduceSum(axis=1, output_dtype=Rfp16(eb=-15))(input_0):runOn(chip=0, cluster=0)) that already specifies a compilable output dtype Rfp16:eb(-15),even though conversion config is specified.


CompilationOutput attrs: ['pass_infos', 'tdx_path']

Debug artifacts written to: /tmp/tmpy8ynb9kh/debug
  00_graph_lowering_pass/graph.txt
  01_compiler_compilation_pass/graph.txt
  02_constant_folding_pass/graph.txt
  03_constant_mapping_pass/graph.txt


## Step 4 — Count and inspect the VpuKernel subgraphs

In [4]:
# Read the debug artifacts directly — these are text dumps of the IR
# at each compilation pass.

for txt in sorted(Path(debug_path).rglob('graph.txt')):
    print(f'\n{"=" * 70}')
    print(f'  {txt.relative_to(debug_path)}')
    print(f'{"=" * 70}')
    print(txt.read_text())



  00_graph_lowering_pass/graph.txt
main = RootGraph()
    input_0 = Rfp16:eb(-15){4, 32}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (4, 32)), on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))()
    _G4 = GroupGraph()
        _G53 = GroupGraph()
            _T54 = Rfp16:eb(-15){32, 4}:AMEM << compiler.Transpose(axes=(1, 0))(input_0):runOn(chip=0, cluster=0)
        _G6 = VpuKernel()
            split_acc2_acc_init = Rfp21:eb(-15){1, 4}:AMEM << compiler.LoadZero(output_type=Type(ir.Rfp21(eb=-15), (1, 4)))():runOn(chip=0, cluster=0)
            _G8 = RepeatGraph(times=1)
                split_acc2_s1_acc_init = Rfp21:eb(-15){1, 4}:AMEM << compiler.LoadZero(output_type=Type(ir.Rfp21(eb=-15), (1, 4)))():runOn(chip=0, cluster=0)
                _G10 = RepeatGraph(times=3)
                    split_acc2_s1_s0_acc_init = Rfp21:eb(-15){1, 4}:AMEM << compiler.LoadZero(output_type=Type(ir.Rfp21(eb=-15), (1, 4)))():runOn(chip=0, cluster=0)
                    _G12 = RepeatG

## Step 5 — Try a slightly bigger reduce (axis=last on a wider row)

Use a row that's wide enough to need multiple iterations on the VPU. This will show RepeatGraph clearly inside the VpuKernel.

In [5]:
class WideReduceModel(nn.Module):
    def build(self, x: nn.Tensor) -> nn.Tensor:
        return nn.reduce_sum(x, axis=-1, keep_dim=True)

wide_model = WideReduceModel()

with tempfile.TemporaryDirectory() as tmp:
    wide_graph = tracing.trace(
        wide_model,
        (nn.Tensor(shape=(8, 256), dtype=ir.Rfp16(-15)),),
        constants_path=tmp,
    )

print('=== WIDE REDUCE FRONTEND GRAPH ===')
print(wide_graph)

=== WIDE REDUCE FRONTEND GRAPH ===
main = RootGraph()
    input_0 = Rfp16:eb(-15){8, 256}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (8, 256)), on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))()
    _T2 = Rfp16:eb(-15){8, 1}:AMEM << ReduceSum(axis=1, output_dtype=Rfp16(eb=-15))(input_0):runOn(chip=0, cluster=0)
    output_0 = Rfp16:eb(-15){8, 1}:AMEM << compiler.Output(on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))(_T2)


In [6]:
wide_conv = sdk.convert(wide_graph)
wide_lowered = wide_conv.graph

print('=== WIDE REDUCE LOWERED GRAPH ===')
print(wide_lowered)

wide_kernels = [sg for sg in wide_lowered.iterate_subgraphs() if isinstance(sg, VpuKernel)]
print(f'\nNumber of VpuKernels: {len(wide_kernels)}')


Skipping op _T2 (_T2 = Rfp16:eb(-15){8, 1}:AMEM << ReduceSum(axis=1, output_dtype=Rfp16(eb=-15))(input_0):runOn(chip=0, cluster=0)) that already specifies a compilable output dtype Rfp16:eb(-15),even though conversion config is specified.


=== WIDE REDUCE LOWERED GRAPH ===
main = RootGraph()
    input_0 = Rfp16:eb(-15){8, 256}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (8, 256)), on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))()
    _T2 = Rfp16:eb(-15){8, 1}:AMEM << ReduceSum(axis=1, output_dtype=Rfp16(eb=-15))(input_0):runOn(chip=0, cluster=0)
    output_0 = Rfp16:eb(-15){8, 1}:AMEM << compiler.Output(on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))(_T2)

Number of VpuKernels: 0


## Step 6 — Compare to the direct-IR-emission target

This is the format the agent emits in the VPU extension — no `nn.Module`, no tracing, just construct the `ir.Graph` by hand:

In [7]:
# Pattern from python/e2e-test-gen/tests/vpu/add_test.py

def build_add_graph(input_type: ir.Type) -> ir.RootGraph:
    graph = ir.RootGraph()
    vpu_type = ir.Type(input_type.dtype, tuple(input_type.shape))

    input_0 = graph.insert(ir.instructions.Input(output_type=input_type))
    input_1 = graph.insert(ir.instructions.Input(output_type=input_type))

    with ir.VpuKernel(parent=graph) as vpu_kernel:
        load_0 = vpu_kernel.insert(ir.instructions.pyxis.vpu.Load(), args=(input_0,), output_type=vpu_type)
        load_1 = vpu_kernel.insert(ir.instructions.pyxis.vpu.Load(), args=(input_1,), output_type=vpu_type)
        add_0  = vpu_kernel.insert(ir.instructions.pyxis.vpu.Add(),  args=(load_0, load_1), output_type=vpu_type)

    graph.insert(ir.instructions.Output(), args=(add_0,), output_type=vpu_type)
    return graph


input_type = ir.Type(ir.Rfp16(-15), (4, 1, 16))
manual_graph = build_add_graph(input_type)
print("=== MANUAL ir.Graph (no nn.Module) ===")
print(manual_graph)

=== MANUAL ir.Graph (no nn.Module) ===
main = RootGraph()
    _T1 = Rfp16:eb(-15){4, 1, 16}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (4, 1, 16)))()
    _T2 = Rfp16:eb(-15){4, 1, 16}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (4, 1, 16)))()
    _G3 = VpuKernel()
        _T4 = Rfp16:eb(-15){4, 1, 16}:AMEM << compiler.Load()(_T1):runOn(chip=0, cluster=0)
        _T5 = Rfp16:eb(-15){4, 1, 16}:AMEM << compiler.Load()(_T2):runOn(chip=0, cluster=0)
        _T6 = Rfp16:eb(-15){4, 1, 16}:AMEM << compiler.Add()(_G3._T4, _G3._T5):runOn(chip=0, cluster=0)
    _T7 = Rfp16:eb(-15){4, 1, 16}:AMEM << compiler.Output()(_G3._T6)


## Step 7 — A simpler VPU kernel: `nn.add`

ReduceSum needed split-accumulation (3 nested RepeatGraphs). Plain `add` is much simpler — element-wise, no accumulator. This shows what the agent's target output should look like for trivial ops.

In [8]:
class AddModel(nn.Module):
    def build(self, x: nn.Tensor, y: nn.Tensor) -> nn.Tensor:
        return nn.add(x, y)

add_model = AddModel()

with tempfile.TemporaryDirectory() as tmp:
    add_graph = tracing.trace(
        add_model,
        (
            nn.Tensor(shape=(4, 32), dtype=ir.Rfp16(-15)),
            nn.Tensor(shape=(4, 32), dtype=ir.Rfp16(-15)),
        ),
        constants_path=tmp,
    )

add_conv = sdk.convert(add_graph)

tmp_dir = tempfile.mkdtemp()
add_debug = Path(tmp_dir) / 'debug'
sdk.compile(
    graph=add_conv.graph,
    conversion_config=add_conv.conversion_config,
    target=Target.Pyxis,
    tdx_path=Path(tmp_dir) / 'add.tdx',
    debug_artifacts_path=add_debug,
)

print((add_debug / '00_graph_lowering_pass' / 'graph.txt').read_text())

Skipping op _T3 (_T3 = Rfp16:eb(-15){4, 32}:AMEM << Add()(input_0, input_1):runOn(chip=0, cluster=0)) that already specifies a compilable output dtype Rfp16:eb(-15),even though conversion config is specified.


main = RootGraph()
    input_0 = Rfp16:eb(-15){4, 32}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (4, 32)), on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))()
    input_1 = Rfp16:eb(-15){4, 32}:AMEM << compiler.Input(output_type=Type(ir.Rfp16(eb=-15), (4, 32)), on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))()
    _G5 = GroupGraph()
        _T6 = Auto{}:AMEM << compiler.Interleave(axis=-2, order=(1, 1))(input_0, input_1):runOn(chip=0, cluster=0)
        _G7 = VpuKernel()
            _G8 = RepeatGraph(times=4)
                _T9 = Auto{}:AMEM << compiler.Load()(_G5._T6):runOn(chip=0, cluster=0)
                _T10 = Rfp16:eb(-15){4, 32}:AMEM << compiler.Add()(_G5._G7._G8._T9, _G5._T6):runOn(chip=0, cluster=0)
    output_0 = Rfp16:eb(-15){4, 32}:AMEM << compiler.Output(on_storage=StorageDevice(chip=0, cluster=0, memory=AMEM))(_G5._G7._G8._T10)


## Step 8 — Actually RUN a VPU kernel and see the tensors flow

We can use `runner.transpile_to_torch_runnable()` to execute the IR graph with real tensors. This is a torch-based emulator — not the actual hardware/TLM, but accurate enough to observe what each VPU op produces.

Pattern from `python/tensordyne-runner/tests/integration/vpu_runner_test.py`.

In [9]:
from tensordyne import runner
from tensordyne.runner.testing.numeric import create_random_tensors

# Use Int16 to avoid float emulation issues — this matches add_test.py's simplest case.
input_type = ir.Type(ir.Int16(), (1, 3, 16))
vpu_type = ir.Type(input_type.dtype, tuple(input_type.shape))

manual_graph = ir.RootGraph()
input_0 = manual_graph.insert(ir.instructions.Input(output_type=input_type))

with ir.VpuKernel(parent=manual_graph) as vpu_kernel:
    load_0 = vpu_kernel.insert(ir.instructions.pyxis.vpu.Load(), args=(input_0,), output_type=vpu_type)
    load_1 = vpu_kernel.insert(ir.instructions.pyxis.vpu.Load(), args=(input_0,), output_type=vpu_type)
    load_2 = vpu_kernel.insert(ir.instructions.pyxis.vpu.Load(), args=(input_0,), output_type=vpu_type)
    add_0  = vpu_kernel.insert(ir.instructions.pyxis.vpu.Add(), args=(load_0, load_1), output_type=vpu_type)
    add_1  = vpu_kernel.insert(ir.instructions.pyxis.vpu.Add(), args=(add_0, load_2), output_type=vpu_type)

manual_graph.insert(ir.instructions.Output(), args=(add_1,), output_type=vpu_type)

print('=== MANUAL VPU GRAPH ===')
print(manual_graph)


=== MANUAL VPU GRAPH ===
main = RootGraph()
    _T1 = Int16{1, 3, 16}:AMEM << compiler.Input(output_type=Type(ir.Int16(), (1, 3, 16)))()
    _G2 = VpuKernel()
        _T3 = Int16{1, 3, 16}:AMEM << compiler.Load()(_T1):runOn(chip=0, cluster=0)
        _T4 = Int16{1, 3, 16}:AMEM << compiler.Load()(_T1):runOn(chip=0, cluster=0)
        _T5 = Int16{1, 3, 16}:AMEM << compiler.Load()(_T1):runOn(chip=0, cluster=0)
        _T6 = Int16{1, 3, 16}:AMEM << compiler.Add()(_G2._T3, _G2._T4):runOn(chip=0, cluster=0)
        _T7 = Int16{1, 3, 16}:AMEM << compiler.Add()(_G2._T6, _G2._T5):runOn(chip=0, cluster=0)
    _T8 = Int16{1, 3, 16}:AMEM << compiler.Output()(_G2._T7)


In [10]:
import torch
from tensordyne.runner.testing.numeric import create_random_tensors

# Run it with a real tensor
input_tensors = create_random_tensors([input_type], seed=42)
print(f'Input shape: {input_tensors[0].shape}')
print(f'Input (first row): {input_tensors[0][0, 0]}')

runnable = runner.transpile_to_torch_runnable(manual_graph)
output = runnable(*input_tensors)

# runnable returns a tuple; unwrap if single output
if isinstance(output, tuple):
    output = output[0]

print(f'\nOutput shape: {output.shape}')
print(f'Output (first row): {output[0, 0]}')

expected = 3 * input_tensors[0]
print(f'\nExpected (3*input, first row): {expected[0, 0]}')

max_diff = (output - expected).abs().max().item()
print(f'\nMax difference vs 3*input: {max_diff:.2e}')


Input shape: torch.Size([1, 3, 16])
Input (first row): tensor([ 6, 35, 28, 46, 42, 23, 28, 20, 22, 41, 34, 22, 26, 42, 39,  4],
       dtype=torch.int16)

Output shape: torch.Size([1, 1, 16])
Output (first row): tensor([ 84,  67,  72, 108,  90,  51,  45,  85,  59,  84,  64,  70,  84,  96,
        122,  50], dtype=torch.int16)

Expected (3*input, first row): tensor([ 18, 105,  84, 138, 126,  69,  84,  60,  66, 123, 102,  66,  78, 126,
        117,  12], dtype=torch.int16)

Max difference vs 3*input: 8.80e+01


## Step 9 — Verify what `_T6` means in `Add(_T9, _T6)`

Build the exact compiler-generated graph and feed deterministic values to see if:
- `_T6` triggers a 2nd FIFO pop (output = row_0 of input_0 + row_0 of input_1)
- `_T6` broadcasts the loaded value (output = 2 × row_0)
- something else

In [11]:
import torch
from tensordyne import runner

# Use Int16 with deterministic values so we can read the result clearly.
shape = (4, 4)
input_type = ir.Type(ir.Int16(), shape)
vpu_type = ir.Type(ir.Int16(), shape)

g = ir.RootGraph()
input_0 = g.insert(ir.instructions.Input(output_type=input_type))
input_1 = g.insert(ir.instructions.Input(output_type=input_type))

# Step 1: Interleave the two inputs (this is OUTSIDE the VpuKernel).
interleaved_shape = (8, 4)  # 4+4 rows alternating
interleaved_type = ir.Type(ir.Int16(), interleaved_shape)
interleaved = g.insert(
    ir.instructions.Interleave(axis=-2, order=(1, 1)),
    args=(input_0, input_1),
    output_type=interleaved_type,
)

# Step 2: VpuKernel with RepeatGraph(4) inside, doing Load + Add(_T9, interleaved).
with ir.VpuKernel(parent=g) as vpu_kernel:
    with ir.RepeatGraph(parent=vpu_kernel, times=4) as repeat:
        loaded = repeat.insert(
            ir.instructions.pyxis.vpu.Load(),
            args=(interleaved,),
            output_type=vpu_type,
        )
        added = repeat.insert(
            ir.instructions.pyxis.vpu.Add(),
            args=(loaded, interleaved),
            output_type=vpu_type,
        )

g.insert(ir.instructions.Output(), args=(added,), output_type=vpu_type)

print('=== EXPERIMENT GRAPH ===')
print(g)

=== EXPERIMENT GRAPH ===
main = RootGraph()
    _T1 = Int16{4, 4}:AMEM << compiler.Input(output_type=Type(ir.Int16(), (4, 4)))()
    _T2 = Int16{4, 4}:AMEM << compiler.Input(output_type=Type(ir.Int16(), (4, 4)))()
    _T3 = Int16{8, 4}:AMEM << compiler.Interleave(axis=-2, order=(1, 1))(_T1, _T2):runOn(chip=0, cluster=0)
    _G4 = VpuKernel()
        _G5 = RepeatGraph(times=4)
            _T6 = Int16{4, 4}:AMEM << compiler.Load()(_T3):runOn(chip=0, cluster=0)
            _T7 = Int16{4, 4}:AMEM << compiler.Add()(_G4._G5._T6, _T3):runOn(chip=0, cluster=0)
    _T8 = Int16{4, 4}:AMEM << compiler.Output()(_G4._G5._T7)


In [12]:
# Run with deterministic values
in0 = torch.tensor([
    [1, 1, 1, 1],
    [2, 2, 2, 2],
    [3, 3, 3, 3],
    [4, 4, 4, 4],
], dtype=torch.int16)
in1 = torch.tensor([
    [10, 10, 10, 10],
    [20, 20, 20, 20],
    [30, 30, 30, 30],
    [40, 40, 40, 40],
], dtype=torch.int16)

print('Input 0:')
print(in0)
print('\nInput 1:')
print(in1)

runnable = runner.transpile_to_torch_runnable(g)
out = runnable(in0, in1)
if isinstance(out, tuple):
    out = out[0]

print(f'\nOutput shape: {out.shape}')
print(f'Output:\n{out}')

print('\n--- Hypothesis check ---')
print(f'If _T6 = FIFO pop:    expect [11,22,33,44] per element  → {(in0 + in1)}')
print(f'If _T6 = broadcast:   expect [2,4,6,8] per element       → {(in0 + in0)}')

Input 0:
tensor([[1, 1, 1, 1],
        [2, 2, 2, 2],
        [3, 3, 3, 3],
        [4, 4, 4, 4]], dtype=torch.int16)

Input 1:
tensor([[10, 10, 10, 10],
        [20, 20, 20, 20],
        [30, 30, 30, 30],
        [40, 40, 40, 40]], dtype=torch.int16)

Output shape: torch.Size([4, 4])
Output:
tensor([[11, 11, 11, 11],
        [22, 22, 22, 22],
        [33, 33, 33, 33],
        [44, 44, 44, 44]], dtype=torch.int16)

--- Hypothesis check ---
If _T6 = FIFO pop:    expect [11,22,33,44] per element  → tensor([[11, 11, 11, 11],
        [22, 22, 22, 22],
        [33, 33, 33, 33],
        [44, 44, 44, 44]], dtype=torch.int16)
If _T6 = broadcast:   expect [2,4,6,8] per element       → tensor([[2, 2, 2, 2],
        [4, 4, 4, 4],
        [6, 6, 6, 6],
        [8, 8, 8, 8]], dtype=torch.int16)


In [ ]:
# Helper used by Step 10 and Step 11 — counts kernels, repeat-graphs, transposes.
def summarize(graph: ir.RootGraph) -> None:
    from tensordyne.ir.vpu import VpuKernel
    from tensordyne.ir.repeat import RepeatGraph

    n_kernels = sum(1 for sg in graph.iterate_subgraphs() if isinstance(sg, VpuKernel))
    n_repeats = sum(1 for sg in graph.iterate_subgraphs() if isinstance(sg, RepeatGraph))
    transpose_ops = [
        op for op in graph.iterate_ops()
        if "Transpose" in type(op.instruction).__name__
    ]
    print(f"VpuKernels:    {n_kernels}")
    print(f"RepeatGraphs:  {n_repeats}")
    print(f"Transposes:    {len(transpose_ops)}")

## Step 10 — Inspect the agent's generated softmax as an IR graph

Reading the agent's `build_graph(...)` Python source is hard. The IR graph print-out is much more concise: it shows each VpuKernel, each RepeatGraph, and each op with its inputs and types on one line. Use this to skim "what did the agent actually emit" without parsing the build code.

Path: `tests/outputs/tensordyne_softmax_kernel_vpu_trial_5.py`

In [ ]:
import importlib.util
from pathlib import Path

import tensordyne.ir as ir


def load_agent_build_graph(path: str | Path):
    """Load a `build_graph` function from an agent-generated output file."""
    path = Path(path)
    spec = importlib.util.spec_from_file_location(path.stem, path)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod.build_graph


AGENT_OUT = "/workspaces/rock/tests/outputs/tensordyne_softmax_kernel_vpu_trial_5.py"
build_graph = load_agent_build_graph(AGENT_OUT)

# Use the same shape the agent was run on so it's a fair comparison with Step 11.
input_type = ir.Type(ir.Rfp16(-15), (4, 16))
softmax_graph = build_graph(input_type)

print("=== AGENT-GENERATED SOFTMAX — IR GRAPH ===")
print(softmax_graph)
print()
summarize(softmax_graph)

## Step 11 — Engineer's softmax IR (compiler-produced via `lower_softmax`)

Drive `nn.softmax` through `sdk.compile` with debug artifacts on, then read the lowering-pass output. This is what the engineering codebase produces — for direct side-by-side comparison with the agent's hand-written IR (Step 10).

Important: VPU lowering only kicks in when the dtype is Rfp16 (or another 16-bit dtype). Fp32 input falls back to a generic non-VPU lowering. We use Rfp16(-15) to match the agent.

In [ ]:
# Attach a SoftmaxLoweringConfig to the Softmax HLI before compile.
# nn.softmax doesn't set this automatically; the compile pipeline requires it
# to know the output EB and to know that we want the VPU lowering path
# (vs. the generic IEEE lowering for Fp32 inputs).
import numpy as np
from tensordyne.ir import filters as ir_filters

eb = ENG_DTYPE.eb  # -15
recip_lut_values_raw = ir.TensorLiteral(
    ir.Rfp16(-15),
    np.full((257, 2), fill_value=np.nan, dtype=np.uint16),  # filled in by pipeline pass
    is_raw=True,
)
softmax_cfg = ir.instructions.SoftmaxLoweringConfig(
    recip_lut_values_raw=recip_lut_values_raw,
    sum_out_type=ir.Rfp16(eb=eb),
)
for op in ir_filters.filter_by_instruction_type(engineer_graph, ir.instructions.Softmax):
    op.instruction.lowering_config = softmax_cfg

# Compile through the full pipeline and capture intermediate IR dumps.
eng_conv = sdk.convert(engineer_graph)

tmp_dir   = tempfile.mkdtemp()
eng_debug = Path(tmp_dir) / 'debug'
sdk.compile(
    graph=eng_conv.graph,
    conversion_config=eng_conv.conversion_config,
    target=Target.Pyxis,
    tdx_path=Path(tmp_dir) / 'softmax.tdx',
    debug_artifacts_path=eng_debug,
    verbose=False,
)

print("=== ENGINEER'S SOFTMAX — POST-LOWERING IR ===")
print((eng_debug / '00_graph_lowering_pass' / 'graph.txt').read_text())

In [ ]:
# Compile through the full pipeline and capture intermediate IR dumps.
# The graph_lowering_pass dump is what we want — that's where `lower_softmax`
# turns the high-level Softmax instruction into VPU kernels.

eng_conv = sdk.convert(engineer_graph)

tmp_dir   = tempfile.mkdtemp()
eng_debug = Path(tmp_dir) / 'debug'
sdk.compile(
    graph=eng_conv.graph,
    conversion_config=eng_conv.conversion_config,
    target=Target.Pyxis,
    tdx_path=Path(tmp_dir) / 'softmax.tdx',
    debug_artifacts_path=eng_debug,
    verbose=False,
)

print("=== ENGINEER'S SOFTMAX — POST-LOWERING IR ===")
print((eng_debug / '00_graph_lowering_pass' / 'graph.txt').read_text())